# Streaming Data Ingestion 

In [0]:
# bronze fact stream notebook
from pyspark.sql.functions import *

EH_NAMESPACE = "enp-strmorders-eus-dev-0526"
EH_NAME = "eh-eus-dev-0526"
EH_CONN_SHARED_ACCESS_KEY_NAME = "RootManageSharedAccessKey"
SECRET_SCOPE = "adb-dev-scope"

EH_CONN_SHARED_ACCESS_KEY_VALUE = dbutils.secrets.get(
    scope=SECRET_SCOPE,
    key=EH_CONN_SHARED_ACCESS_KEY_NAME
)

EH_CONN_STR = f"Endpoint=sb://{EH_NAMESPACE}.servicebus.windows.net/;SharedAccessKeyName={EH_CONN_SHARED_ACCESS_KEY_NAME};SharedAccessKey={EH_CONN_SHARED_ACCESS_KEY_VALUE}"

KAFKA_OPTIONS = {
    "kafka.bootstrap.servers": f"{EH_NAMESPACE}.servicebus.windows.net:9093",
    "subscribe": EH_NAME,
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{EH_CONN_STR}";',
    "startingOffsets": "earliest",
    "failOnDataLoss": "false"
}

df = (
    spark.readStream
    .format("kafka")
    .options(**KAFKA_OPTIONS)
    .load()
)

df_parsed = df.selectExpr(
    "CAST(value AS STRING) as json_data",
    "timestamp"
)

bronze_checkpoint_path = "abfss://bronze-layer@steusadlesgen0526.dfs.core.windows.net/orders/checkpoints"
bronze_path = "abfss://bronze-layer@steusadlesgen0526.dfs.core.windows.net/orders"

# IMPORTANT: clear checkpoint if changed query
# dbutils.fs.rm(bronze_checkpoint_path, True)

query = (
    df_parsed.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", bronze_checkpoint_path)
    .trigger(processingTime="1 minute")
    .start(bronze_path)
)

from datetime import datetime, timedelta
import time

end_time = datetime.now() + timedelta(hours=12)

while datetime.now() < end_time:
    if not query.isActive:
        break
    time.sleep(60)

query.stop()


In [0]:
# display(
#     df_parsed,
#     checkpointLocation="abfss://bronze-layer@steusadlesgen0526.dfs.core.windows.net/orders/display_checkpoint"
# )